In [1]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import json
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
#DataStore store stores the samples and their corresponding labels
#DataLoader wraps an iterable around the Dataset to enable easy access to the samples.


In [3]:

"""
for i, (session_id, eventstotal) in enumerate(objects):
    print(f"Session :", session_id)
    print(f"event number :", eventstotal)
    
    for singular_event in eventstotal:
        print(datetime.datetime.fromtimestamp(int(singular_event['ts'] / 1000)).strftime('%d-%m-%Y %H:%M:%S'))S
    if i == 1:
        break
        """

'\nfor i, (session_id, eventstotal) in enumerate(objects):\n    print(f"Session :", session_id)\n    print(f"event number :", eventstotal)\n    \n    for singular_event in eventstotal:\n        print(datetime.datetime.fromtimestamp(int(singular_event[\'ts\'] / 1000)).strftime(\'%d-%m-%Y %H:%M:%S\'))S\n    if i == 1:\n        break\n        '

In [4]:
def extract_json(filename: str):
    with open(filename, "r") as f:
        for line in f:
            session = json.loads(line)
            yield session["session"], session["events"]

objects = extract_json("train.jsonl")

print(type(objects))


<class 'generator'>


In [7]:
sessions = []

for i, (session_id, eventstotal) in enumerate(extract_json("../../train.jsonl")):
    aids, timestamps, events_type = [], [], []
    for event in eventstotal:
        aids.append(event["aid"])
        timestamps.append(event["ts"])
        events_type.append(event["type"])
        
    sessions.append({
            "session_id": i,
            "events": {
            "aid": aids,
            "timestamps": timestamps,
            "events_type": events_type    
            },
        })
    if i >= 200:
        break


In [8]:
class OttoDataSetSession(Dataset):
    def __init__(self, session):
        self.session = session
        self.event_map = {"clicks":1, "carts": 2, "orders": 3}

    def __len__(self) -> int:
        return len(self.session)


    def __getitem__(self, index):
        session = self.session[index]
                 
        events = session["events"]
        
        aids = torch.tensor(events["aid"], dtype=torch.int64)
        
        timestamps = torch.tensor(events["c"], dtype=torch.long)
        
        events_type = torch.tensor( [self.event_map[e] for e in events["events_type"]], dtype=torch.int64)
        return {
            "session_id": torch.tensor(session["session_id"], dtype=torch.int64),
            "aid": aids,
            "timestamps": timestamps,
            "type": events_type
        }
    

In [9]:
dataset = OttoDataSetSession(sessions)

#To see the samples
#for i in range(0,1):
#    sample = dataset[i]
#    print(sample)


In [10]:
def custom_collate(batch):
    aids = [torch.tensor(d["aid"]) for d in batch]
    timestamps = [torch.tensor(d["timestamps"]) for d in batch]
    events_type = [torch.tensor(d["type"]) for d in batch]
    
    aids_padded = pad_sequence(aids, batch_first=True, padding_value=0)
    timestamps_padded = pad_sequence(timestamps, batch_first=True, padding_value=0)
    events_type_padded = pad_sequence(events_type, batch_first=True, padding_value=0)
    
    session_id = [d["session_id"] for d in batch]
    return {
        "aids" : aids_padded,
        "timestamps" : timestamps_padded,
        "events_type" : events_type_padded,
        "session_id" : torch.stack(session_id)
    }
    

In [11]:
train_loader = DataLoader(dataset=dataset, batch_size=64, shuffle=False, collate_fn=custom_collate)
for batch in train_loader:
    print(f"Shape of the Batch Aids:{batch["aids"].shape}, Shape of the Batch Ts:{batch["timestamps"].shape}, Shape of the batch Type:{batch["events_type"].shape}")

KeyError: 'c'